In [1]:
import torch
print(torch.__version__)

2.11.0+cpu


In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader


d:\Data science - Projects\airline-complaint-severity-detection\airline_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

df = pd.read_csv("../data/clean_airline_reviews.csv")



# Create severity label
def severity_label(rating):
    if rating <= 2:
        return "Critical"
    elif rating <= 4:
        return "High"
    elif rating <= 6:
        return "Medium"
    else:
        return "Low"

df["Overall_Rating"] = pd.to_numeric(df["Overall_Rating"], errors="coerce")
df["severity_label"] = df["Overall_Rating"].apply(severity_label)

# Encode to numbers
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["severity_id"] = le.fit_transform(df["severity_label"])

df = df.groupby("severity_id").sample(800, random_state=42)
texts = df["final_text"].astype(str).tolist()


labels = df["severity_id"].tolist()
print(len(texts))
print(len(labels))


3200
3200


In [6]:

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(len(X_train), len(X_test))

# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=64)

# Dataset class
class AirlineDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = AirlineDataset(train_encodings, y_train)
test_dataset = AirlineDataset(test_encodings, y_test)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

2560 640


In [7]:

# Model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)



Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3814.98it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Training
model.train()

for epoch in range(3):
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

print("Training complete")

Training complete


In [9]:
# Evaluation
model.eval()

y_pred = []
y_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels_batch.cpu().numpy())


In [10]:
# Metrics
print("\nDistilBERT Classification Report:\n")
print(classification_report(y_true, y_pred))

bert_accuracy = np.mean(np.array(y_pred) == np.array(y_true))
bert_precision = precision_score(y_true, y_pred, average='weighted')
bert_recall = recall_score(y_true, y_pred, average='weighted')
bert_f1 = f1_score(y_true, y_pred, average='weighted')

print("\nAccuracy:", bert_accuracy)
print("Precision:", bert_precision)
print("Recall:", bert_recall)
print("F1 Score:", bert_f1)


DistilBERT Classification Report:

              precision    recall  f1-score   support

           0       0.59      0.62      0.60       160
           1       0.41      0.35      0.38       160
           2       0.57      0.71      0.63       160
           3       0.49      0.42      0.45       160

    accuracy                           0.52       640
   macro avg       0.51      0.52      0.52       640
weighted avg       0.51      0.52      0.52       640


Accuracy: 0.5234375
Precision: 0.5142890364159557
Recall: 0.5234375
F1 Score: 0.5158701411271149


In [16]:
results_df = pd.read_csv("../models/model_results.csv")

new_result = pd.DataFrame({
    "Model": ["DistilBERT"],
    "Accuracy": [bert_accuracy],
    "Precision": [bert_precision],
    "Recall": [bert_recall],
    "F1 Score": [bert_f1]
})

results_df = pd.concat([results_df, new_result], ignore_index=True)

results_df.to_csv("../models/model_results.csv", index=False)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.646664,0.491140,0.540800,0.504593
1,Naive Bayes,0.717197,0.461949,0.427965,0.376047
2,SVM,0.705777,0.474021,0.442200,0.443481
3,LSTM,0.686731,0.642167,0.686731,0.660004
4,CNN,0.707659,0.626311,0.707659,0.656906
5,DistilBERT,0.712000,0.623899,0.712000,0.658037


In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [7]:
model.save_pretrained("../models/bert_model")
tokenizer.save_pretrained("../models/bert_model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.98s/it]


('../models/bert_model\\tokenizer_config.json',
 '../models/bert_model\\tokenizer.json')